In [10]:
from google.colab import drive
drive.mount('/content/drive')

FOLDERNAME = 'ML/Final/ML_final/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

%cd /content/drive/My\ Drive/$FOLDERNAME

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/ML/Final/ML_final


In [11]:
import os
print("cwd:", os.getcwd())
print(os.listdir())

cwd: /content/drive/MyDrive/ML/Final/ML_final
['README.md', 'model.joblib', 'model_lgb.joblib', 'model_experiment_LightGBM.ipynb', 'EDA.ipynb', 'data_extraction_feature_engineering.ipynb', '.gitignore', 'model_experiment_XGBoost.ipynb', 'dlinear_model.pt', 'model_experiment_DLinear.ipynb', 'lightning_logs', '.idea', '.git', 'data', 'wandb', 'model_experiment_TFT.ipynb']


In [12]:
!jupyter nbconvert --to script data_extraction_feature_engineering.ipynb

import importlib
import data_extraction_feature_engineering as prep
importlib.reload(prep)

df_train_full, df_test_full = prep.extract_data()
VAL_START_DATE = '2012-09-01'

[NbConvertApp] Converting notebook data_extraction_feature_engineering.ipynb to script
[NbConvertApp] Writing 8340 bytes to data_extraction_feature_engineering.py
Loading data...
Train shape: (421570, 5)
Features shape: (8190, 12)
Stores shape: (45, 3)
Test shape: (115064, 4)
Merging datasets...

Data Preparation Complete!
X_train shape: (397841, 24)
X_val shape: (23729, 24)
   IsHoliday  Temperature  Fuel_Price  MarkDown1  MarkDown2  MarkDown3  \
0      False        42.31       2.572        0.0        0.0        0.0   
1       True        38.51       2.548        0.0        0.0        0.0   
2      False        39.93       2.514        0.0        0.0        0.0   
3      False        46.63       2.561        0.0        0.0        0.0   
4      False        46.50       2.625        0.0        0.0        0.0   

   MarkDown4  MarkDown5         CPI  Unemployment  ...  sales_lag_52  \
0        0.0        0.0  211.096358         8.106  ...           NaN   
1        0.0        0.0  211.2421

In [13]:
!pip install wandb -q
import wandb
import os
# Retrieve the secret from Kaggle Secrets
api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [14]:
!pip install pytorch-forecasting lightning -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 21.9 MB/s eta 0:00:00


In [16]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
df = prep.create_features1(df_train_full, drop_columns=['Id'])
df['Weekly_Sales'] = df['Weekly_Sales'].clip(lower=0)
df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

df['group_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
df['time_idx'] = df.groupby('group_id')['Date'].rank(method='first').astype(int) - 1

for col in ['Store', 'Dept', 'Type', 'IsHoliday']:
    if col in df.columns:
        df[col] = df[col].astype(str)

markdown_cols = [c for c in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5'] if c in df.columns]

print(df[['Store', 'Dept', 'group_id', 'time_idx', 'Date']].head())
print(f"Series count: {df['group_id'].nunique()}")

  Store Dept group_id  time_idx       Date
0     1    1      1_1         0 2010-02-05
1     1    1      1_1         1 2010-02-12
2     1    1      1_1         2 2010-02-19
3     1    1      1_1         3 2010-02-26
4     1    1      1_1         4 2010-03-05
Series count: 3331


## 2. Defining TimeSeriesDataSet

In [17]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

MAX_ENCODER_LENGTH = 52
MAX_PREDICTION_LENGTH = 8

training_cutoff_idx = df.loc[df['Date'] < VAL_START_DATE, 'time_idx'].max()

training = TimeSeriesDataSet(
    df[df.time_idx <= training_cutoff_idx],
    time_idx='time_idx',
    target='Weekly_Sales',
    group_ids=['group_id'],
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=['Store', 'Dept', 'Type'],
    static_reals=['Size'],
    time_varying_known_categoricals=['IsHoliday'],
    time_varying_known_reals=['time_idx', 'Month', 'WeekOfYear'],
    time_varying_unknown_reals=['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment'] + markdown_cols,
    target_normalizer=GroupNormalizer(groups=['group_id']),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)

BATCH_SIZE = 64
train_dataloader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)

print(f"Train windows: {len(training)}, Val windows: {len(validation)}")

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 360 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__group_id': '10_47'}, {'__group_id__group_id': '10_51'}, {'__group_id__group_id': '10_77'}, {'__group_id__group_id': '10_78'}, {'__group_id__group_id': '11_47'}, {'__group_id__group_id': '11_48'}, {'__group_id__group_id': '11_50'}, {'__group_id__group_id': '11_51'}, {'__group_id__group_id': '11_77'}, {'__group_id__group_id': '11_78'}]
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 360 series/groups which therefore are not present i

Train windows: 218587, Val windows: 2971


## 3. Building and training the model

In [18]:
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import WandbLogger

CONFIG = {
    "model_type": "TFT",
    "run_name": "TFT_baseline",
    "max_encoder_length": MAX_ENCODER_LENGTH,
    "max_prediction_length": MAX_PREDICTION_LENGTH,
    "hidden_size": 16,
    "attention_head_size": 2,
    "dropout": 0.1,
    "hidden_continuous_size": 8,
    "learning_rate": 0.03,
    "batch_size": BATCH_SIZE,
    "max_epochs": 30,
    "early_stopping_patience": 5,
    "seed": 42,
}

run = wandb.init(
    project="walmart-sales-forecasting",
    group="TFT_Training",
    name=CONFIG["run_name"],
    job_type="train",
    config=CONFIG,
)
wandb_logger = WandbLogger(experiment=run, log_model=False)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=CONFIG["learning_rate"],
    hidden_size=CONFIG["hidden_size"],
    attention_head_size=CONFIG["attention_head_size"],
    dropout=CONFIG["dropout"],
    hidden_continuous_size=CONFIG["hidden_continuous_size"],
    loss=QuantileLoss(),
    log_interval=10,
    optimizer="Adam",
)
print(f"TFT parameters: {sum(p.numel() for p in tft.parameters()):,}")

early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=CONFIG["early_stopping_patience"], mode="min"
)

trainer = pl.Trainer(
    max_epochs=CONFIG["max_epochs"],
    accelerator="auto",
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback],
    logger=wandb_logger,
    enable_progress_bar=True,
)

trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

TFT parameters: 31,435


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  1.9 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    288 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  2.9 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 10.2 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  2.5 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    119 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 31.4 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 31.4 K                                                                                               
Total estimated model params size (MB): 0.126                                                                      
Modules in train mode: 506                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

## 4. ვალიდაცია (WMAE მეტრიკზე აქაც)

In [20]:
from sklearn.metrics import mean_absolute_error

def compute_wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

raw_predictions = tft.predict(val_dataloader, mode="raw", return_x=True, return_index=True)
median_idx = tft.loss.quantiles.index(0.5)
val_preds = raw_predictions.output.prediction[..., median_idx].cpu().numpy().flatten()

val_index = raw_predictions.index
val_actuals = []
val_is_holiday = []
for i, row in val_index.iterrows():
    series_df = df[(df['group_id'] == row['group_id'])].sort_values('time_idx')
    target_slice = series_df[series_df['time_idx'].between(row['time_idx'], row['time_idx'] + MAX_PREDICTION_LENGTH - 1)]
    val_actuals.extend(target_slice['Weekly_Sales'].values)
    val_is_holiday.extend((target_slice['IsHoliday'] == 'True').astype(int).values)

val_actuals = np.array(val_actuals[:len(val_preds)])
val_is_holiday = np.array(val_is_holiday[:len(val_preds)])

val_mae = mean_absolute_error(val_actuals, val_preds)
val_wmae = compute_wmae(val_actuals, val_preds, val_is_holiday)

print(f"Final Validation MAE:  {val_mae:.2f}")
print(f"Final Validation WMAE: {val_wmae:.2f}")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Final Validation MAE:  2073.19
Final Validation WMAE: 2288.54


## 5. დალოგვა

In [21]:
if run:
    wandb.log({"final_validation_wmae": val_wmae, "final_validation_mae": val_mae})

    plt.figure(figsize=(10, 6))
    order = np.argsort(val_actuals)
    plt.plot(val_actuals[order], label="Actual", alpha=0.7)
    plt.plot(val_preds[order], label="Predicted (median)", alpha=0.7)
    plt.title("TFT: Validation Predictions vs Actuals (sorted by actual value)")
    plt.legend()
    plt.tight_layout()
    wandb.log({"val_predictions_plot": wandb.Image(plt)})
    plt.close()

    interpretation_batch = next(iter(val_dataloader))
    raw_out = tft(interpretation_batch[0])
    interpretation = tft.interpret_output(raw_out, reduction="sum")
    fig_dict = tft.plot_interpretation(interpretation)
    for name, fig in fig_dict.items():
        wandb.log({f"tft_interpretation_{name}": wandb.Image(fig)})
        plt.close(fig)

    trainer.save_checkpoint("tft_model.ckpt")
    artifact = wandb.Artifact("tft-model-final", type="model")
    artifact.add_file("tft_model.ckpt")
    run.log_artifact(artifact)

if run:
    wandb.finish()

INFO: `weights_only` was not set, defaulting to `False`.
INFO:lightning.pytorch.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.


epoch,▁▁▁▁▁▂▂▂▂▂▄▄▄▄▅▅▅▅▅▅▇▇▇▇▇▇▇▇▇▇▇█████████
final_validation_mae,▁
final_validation_wmae,▁
train_loss_epoch,█▄▃▂▂▁
train_loss_step,▃▃▂▃▂▁▃▃▂▄▂▁▂▃▂▅▂▂▂▂▁▃▃▄█▆▅▃▄▃▃▂▁▂▃▃▃▃▃▄
trainer/global_step,▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
val_MAE,▁▄▄█▃▄
val_MAPE,▁▅▆█▆▅
val_RMSE,█▂▂▄▁▁
val_SMAPE,▁▅▅█▅▆
+1,...
